In [0]:
silver_path = "abfss://silver@sgcustomerpipeline.dfs.core.windows.net/SalesLT/"

dbutils.fs.ls(silver_path)

In [0]:
gold_path ="abfss://gold@sgcustomerpipeline.dfs.core.windows.net/"

dbutils.fs.ls(gold_path)

In [0]:
df = spark.read.format('delta').load(silver_path + 'Address')

display(df)

In [0]:
import re

def rename_columns_to_snake_case(df):
    new_columns = []
    for col in df.columns:
        new_name = re.sub(r'(?<!^)(?=[A-Z])', '_', col).lower()
        new_columns.append(new_name)
    return df.toDF(*new_columns)
    

In [0]:
df = rename_columns_to_snake_case(df)
display(df)

In [0]:
table_name_temp = []

for i in dbutils.fs.ls(silver_path):
    table_name_temp.append(i.name.split('/')[0])

In [0]:
table_name_temp

In [0]:
for name in table_name_temp:
    path = silver_path + name
    print(path)

    df = spark.read.format('delta').load(path)
    df = rename_columns_to_snake_case(df)
    
    output_path = gold_path+ 'SalesLT/' + name + '/'
    df.write.format("delta").mode("overwrite").save(output_path)
    